In [1]:
# pyast_config.py (No changes needed, keeping for completeness)
import logging
from typing import List, Dict, Any, Optional, Union, Tuple

# Configure logging for ast_config.py
logger = logging.getLogger(__name__)
# logger.setLevel(logging.DEBUG) # Set to DEBUG for detailed logs, INFO for less verbose

# Helper function for splitting by operator outside parentheses
def _split_by_operator_outside_parens(text: str, operator: str) -> List[str]:
    parts = []
    paren_level = 0
    current_part_start = 0
    op_len = len(operator)

    i = 0
    while i < len(text):
        if text[i] == '(':
            paren_level += 1
        elif text[i] == ')':
            paren_level -= 1
        
        # Check for operator match
        if (i + op_len <= len(text) and
            paren_level == 0 and
            text[i:i+op_len] == operator):
            
            part = text[current_part_start:i].strip()
            if part:
                parts.append(part)
            current_part_start = i + op_len
            i += op_len - 1 # Adjust i to continue after the operator
        i += 1
    
    # Add the last part
    last_part = text[current_part_start:].strip()
    if last_part:
        parts.append(last_part)
    
    return [p for p in parts if p] # Filter out empty strings


# Base AST Node for configuration
class ASTConfigNode:
    """Base class for all AST configuration nodes."""
    def __repr__(self):
        # Default __repr__ calls _repr_indent with level 0
        return self._repr_indent(0)

    def _repr_indent(self, level=0):
        """Generates a readable, indented representation of the config node."""
        indent = "  " * level
        return f"{indent}[{self.__class__.__name__}]" # Default, to be overridden if needed

# --- Main Environment Configuration ---
class GymEnvironmentConfig(ASTConfigNode):
    def __init__(self, env_name: str,
                 definitions: Optional['DefinitionsConfig'] = None,
                 variables: Optional['VariablesConfig'] = None,
                 problem: Optional['ProblemConfig'] = None):
        self.env_name = env_name
        self.definitions = definitions if definitions is not None else DefinitionsConfig()
        self.variables = variables if variables is not None else VariablesConfig()
        self.problem = problem if problem is not None else ProblemConfig()

    def _repr_indent(self, level=0):
        indent = "  " * level
        return (f"{indent}[GymEnvironmentConfig]({self.env_name})\n"
                f"{self.definitions._repr_indent(level + 1)}\n"
                f"{self.variables._repr_indent(level + 1)}\n"
                f"{self.problem._repr_indent(level + 1)}")

# --- Definitions Block Configuration ---
class DefinitionsConfig(ASTConfigNode):
    def __init__(self, constants: Optional[List['ConstantDefinitionConfig']] = None,
                 functions: Optional[List['FunctionDefinitionConfig']] = None,
                 predicates: Optional[List['PredicateDefinitionConfig']] = None,
                 hybrid_programs: Optional[List['HybridProgramConfig']] = None):
        self.constants = constants if constants is not None else []
        self.functions = functions if functions is not None else []
        self.predicates = predicates if predicates is not None else []
        self.hybrid_programs = hybrid_programs if hybrid_programs is not None else [] # Can contain ControlFuncConfig, StepFuncConfig

    def _repr_indent(self, level=0):
        indent = "  " * level
        constants_str = "\n".join([c._repr_indent(level + 4) for c in self.constants])
        functions_str = "\n".join([f._repr_indent(level + 4) for f in self.functions])
        predicates_str = "\n".join([p._repr_indent(level + 4) for p in self.predicates])
        hybrid_programs_str = "\n".join([hp._repr_indent(level + 4) for hp in self.hybrid_programs])

        return (f"{indent}[DefinitionsConfig]\n"
                f"{indent}  [Constants]:\n{constants_str or f'{indent}    (None)'}\n"
                f"{indent}  [Functions]:\n{functions_str or f'{indent}    (None)'}\n"
                f"{indent}  [Predicates]:\n{predicates_str or f'{indent}    (None)'}\n"
                f"{indent}  [Hybrid Programs]:\n{hybrid_programs_str or f'{indent}    (None)'}")

class ConstantDefinitionConfig(ASTConfigNode):
    def __init__(self, name: str, type: str, value: Optional[str] = None, metadata: Optional[Dict] = None):
        self.name = name
        self.type = type
        self.value = value
        self.metadata = metadata or {}

    def _repr_indent(self, level=0):
        indent = "  " * level
        meta = f", Meta={self.metadata}" if self.metadata else ""
        return f"{indent}[ConstantDefinition]({self.name}): Type='{self.type}', Value='{self.value}'{meta}"

class FunctionDefinitionConfig(ASTConfigNode):
    def __init__(self, name: str, return_type: str, parameters: List[Tuple[str, str]], body: str, metadata: Optional[Dict] = None):
        self.name = name
        self.return_type = return_type
        self.parameters = parameters # List of (type, name) tuples
        self.body = body
        self.metadata = metadata or {}

    def _repr_indent(self, level=0):
        indent = "  " * level
        params = ", ".join([f"{t} {n}" for t, n in self.parameters])
        meta = f", Meta={self.metadata}" if self.metadata else ""
        return f"{indent}[FunctionDefinition]({self.name}): Returns='{self.return_type}', Params=({params}), Body='{self.body}'{meta}"

class PredicateDefinitionConfig(ASTConfigNode):
    def __init__(self, name: str, parameters: List[Tuple[str, str]], body: str, metadata: Optional[Dict] = None):
        self.name = name
        self.parameters = parameters # List of (type, name) tuples
        self.body = body
        self.metadata = metadata or {}

    def _repr_indent(self, level=0):
        indent = "  " * level
        params_str = ", ".join([f"{t} {n}" for t, n in self.parameters])
        meta = f", Meta={self.metadata}" if self.metadata else ""

        body_display = self.body.strip()
        # Apply formatting for readability if it contains logical operators
        # This is a heuristic to match the expected multi-line output for predicates
        if '&' in body_display or '|' in body_display:
            # Split by '&' first, then by '|' within each '&' part
            and_parts = _split_by_operator_outside_parens(body_display, '&')
            
            formatted_body_parts = []
            for i, part in enumerate(and_parts):
                if i > 0: # Add newline and indent for subsequent '&' parts
                    formatted_body_parts.append(f"\n{indent}    & ")
                
                # Then, split each 'and_part' by '|'
                or_parts = _split_by_operator_outside_parens(part, '|')
                if len(or_parts) > 1:
                    # If an 'and_part' itself contains '|', format it
                    formatted_sub_part = or_parts[0]
                    for sub_part_idx in range(1, len(or_parts)):
                        if or_parts[sub_part_idx] == '|':
                            formatted_sub_part += f"\n{indent}      | " # Further indent for OR
                        else:
                            formatted_sub_part += or_parts[sub_part_idx]
                    formatted_body_parts.append(formatted_sub_part)
                else:
                    formatted_body_parts.append(part)
            
            body_display = "".join(formatted_body_parts).strip()
        
        # Truncation logic for predicates, if the expected output truncates
        single_line_body_for_len = body_display.replace('\n', ' ').strip()
        if len(single_line_body_for_len) > 80: # Arbitrary threshold for display
            body_display = single_line_body_for_len[:77] + "..."

        return f"{indent}[PredicateDefinition]({self.name}): Params=({params_str}), Body='{body_display}'{meta}"

class HybridProgramConfig(ASTConfigNode):
    """Base class for specific hybrid program configurations."""
    def __init__(self, name: str):
        self.name = name

    def _repr_indent(self, level=0):
        indent = "  " * level
        return f"{indent}[HybridProgram]({self.name})"

class ControlFuncConfig(HybridProgramConfig):
    """Configuration for a control function (discrete choices and assignments)."""
    def __init__(self, name: str, action_space_config: Optional['ActionConfig'] = None, metadata: Optional[Dict] = None):
        super().__init__(name)
        self.action_space_config = action_space_config
        self.metadata = metadata or {}

    def _repr_indent(self, level=0):
        indent = "  " * level
        meta = f" [Meta: {self.metadata}]" if self.metadata else ""

        if self.action_space_config:
            raw_body_snippet = self.action_space_config.to_string_representation()
            if len(raw_body_snippet) > 80:
                raw_body_snippet = raw_body_snippet[:77] + "..."
            return (f"{indent}[HybridProgram]({self.name}){meta}\n"
                    f"{indent}  [Body (parsed)]: '{raw_body_snippet}'\n"
                    f"{self.action_space_config._repr_indent(level + 1)}")
        return f"{indent}[HybridProgram]({self.name}): No action space defined."

class ODEConfig(ASTConfigNode):
    def __init__(self, variable: str, equation: str):
        self.variable = variable
        self.equation = equation

    def _repr_indent(self, level=0):
        indent = "  " * level
        return f"{indent}[ODEConfig(var='{self.variable}', eq='{self.equation}')]"

class EvolutionDomainConfig(ASTConfigNode):
    def __init__(self, constraints: Optional[List[str]] = None):
        self.constraints = constraints if constraints is not None else []

    def _repr_indent(self, level=0):
        indent = "  " * level
        constraints_str = "\n".join([f"{indent}  [Constraint]: '{c}'" for c in self.constraints])
        return f"{indent}[Evolution Domain]:\n{constraints_str or f'{indent}  (None)'}"

class StepFuncConfig(HybridProgramConfig):
    """Configuration for a step function (differential equations)."""
    def __init__(self, name: str, odes: List[ODEConfig], evolution_domain: Optional[EvolutionDomainConfig] = None, duration: Optional[str] = None, raw_body_snippet: Optional[str] = None, metadata: Optional[Dict] = None):
        super().__init__(name)
        self.odes = odes
        self.evolution_domain = evolution_domain
        self.duration = duration # e.g., 'ep' for the t' = 1 & t <= ep part
        self.raw_body_snippet = raw_body_snippet # Added to store raw body for display
        self.metadata = metadata or {}

    def _repr_indent(self, level=0):
        indent = "  " * level

        meta = f" [Meta: {self.metadata}]" if self.metadata else ""
        odes_lines = [ode._repr_indent(level + 2) for ode in self.odes]
        duration_str = f", Duration='{self.duration}'" if self.duration else ""

        ode_block_str = "\n".join(odes_lines) if odes_lines else f"{indent}  (None)"
        domain_block_str = self.evolution_domain._repr_indent(level + 1) if self.evolution_domain else f"{indent}  [Evolution Domain]:\n{indent}    (None)"

        body_parsed_str = ""
        if self.raw_body_snippet:
            body_display = self.raw_body_snippet
            if len(body_display) > 80: # Truncate for display
                body_display = body_display[:77] + "..."
            body_parsed_str = f"{indent}  [Body (parsed)]: '{body_display}'\n"

        return (f"{indent}[HybridProgram]({self.name}){duration_str}{meta}\n"
                f"{body_parsed_str}" # Add this line
                f"{indent}  [ODEs]:\n{ode_block_str}\n"
                f"{domain_block_str}")


# --- Variables Block Configuration ---
class VariablesConfig(ASTConfigNode):
    def __init__(self, state_variables: Optional[List['VariableDeclarationConfig']] = None,
                 auxiliary_variables: Optional[List[str]] = None):
        self.state_variables = state_variables if state_variables is not None else []
        self.auxiliary_variables = auxiliary_variables if auxiliary_variables is not None else [] # Variables not part of observation space

    def _repr_indent(self, level=0):
        indent = "  " * level
        state_vars_str = "\n".join([v._repr_indent(level + 4) for v in self.state_variables])
        aux_vars_str = "\n".join([v._repr_indent(level + 4) for v in self.auxiliary_variables])
        return (f"{indent}[VariablesConfig]\n"
                f"{indent}  [State Variables]:\n{state_vars_str or f'{indent}    (None)'}\n"
                f"{indent}  [Auxiliary Variables]:\n{aux_vars_str or f'{indent}    (None)'}")

class VariableDeclarationConfig(ASTConfigNode):
    def __init__(self, name: str, type: str, lower_bound: Optional[str] = None, upper_bound: Optional[str] = None):
        super().__init__()
        self.name = name
        self.type = type
        self.lower_bound = lower_bound
        self.upper_bound = upper_bound

    def _repr_indent(self, level=0):
        indent = "  " * level
        bounds_str = ""
        if self.lower_bound is not None or self.upper_bound is not None:
            bounds_str = f", Bounds=({self.lower_bound or '-inf'}, {self.upper_bound or 'inf'})"
        return f"{indent}[VariableDeclaration]({self.name}): Type='{self.type}'{bounds_str}"

# --- Problem Statement Configuration ---
class PreconditionConfig(ASTConfigNode):
    def __init__(self, formula: str):
        self.formula = formula
    def _repr_indent(self, level=0):
        indent = "  " * level
        return f"{indent}Precondition(formula='{self.formula}')"

class PostconditionConfig(ASTConfigNode):
    def __init__(self, formula: str):
        self.formula = formula
    def _repr_indent(self, level=0):
        indent = "  " * level
        return f"{indent}Postcondition(formula='{self.formula}')"

class ProblemConfig(ASTConfigNode):
    def __init__(self, precondition: Optional[PreconditionConfig] = None,
                 loop_execution: Optional['LoopExecutionConfig'] = None,
                 postcondition: Optional[PostconditionConfig] = None):
        self.precondition = precondition if precondition is not None else PreconditionConfig(formula="")
        self.loop_execution = loop_execution if loop_execution is not None else LoopExecutionConfig()
        self.postcondition = postcondition if postcondition is not None else PostconditionConfig(formula="")

    def _repr_indent(self, level=0):
        indent = "  " * level
        pre_str = f"{self.precondition._repr_indent(level + 1)}" if self.precondition else ""
        loop_str = f"\n{self.loop_execution._repr_indent(level + 1)}" if self.loop_execution else ""
        post_str = f"\n{self.postcondition._repr_indent(level + 1)}" if self.postcondition else ""
        return (f"{indent}[ProblemConfig]\n"
                f"{indent}  [Precondition]: '{self.precondition.formula}'" # Reverted to match example format
                f"{loop_str}{post_str}")

class LoopExecutionConfig(ASTConfigNode):
    def __init__(self, iterations: str = "inf", loop_invariant_formula: str = "", loop_body: 'SequenceActionConfig' = None):
        self.iterations = iterations
        self.loop_invariant_formula = loop_invariant_formula
        self.loop_body = loop_body if loop_body is not None else SequenceActionConfig() # This will be a SequenceActionConfig containing discrete and continuous parts

    def _repr_indent(self, level=0):
        indent = "  " * level
        return (f"{indent}[LoopExecution]\n"
                f"{indent}  [Iterations]: '{self.iterations}'\n"
                f"{indent}  [LoopInvariant]: '{self.loop_invariant_formula}'\n"
                f"{indent}  [LoopBody]:\n{self.loop_body._repr_indent(level + 2)}")

class HybridProgramStepConfig(ASTConfigNode):
    """Represents a call to a hybrid program within a loop, possibly with an invariant."""
    def __init__(self, hp_name: str, invariant_formula: Optional[str] = None):
        self.hp_name = hp_name
        self.invariant_formula = invariant_formula

    def _repr_indent(self, level=0):
        indent = "  " * level
        invariant_str = ""
        if self.invariant_formula:
            # Format invariant formula with newlines and indentation
            parts = _split_by_operator_outside_parens(self.invariant_formula, ',')
            if len(parts) > 1:
                formatted_invariant = f"@invariant({parts[0]}"
                for part in parts[1:]:
                    formatted_invariant += f",\n{indent}          {part}" # 10 spaces for alignment after indent
                formatted_invariant += ")"
                invariant_str = formatted_invariant
            else:
                invariant_str = f"@invariant({self.invariant_formula})"
        return f"{indent}[HybridProgramStep]: {self.hp_name}{invariant_str}"


# --- Action Space Configuration (for controlFunc and problem body) ---
class ActionConfig(ASTConfigNode):
    """Base class for action configurations."""
    def to_string_representation(self) -> str:
        """Returns a string representation suitable for raw body display."""
        raise NotImplementedError

class ChoiceActionConfig(ActionConfig):
    def __init__(self, level_name: str, choices: List[ActionConfig]):
        self.level_name = level_name # e.g., 'ChoiceLevel_0'
        self.choices = choices

    def _repr_indent(self, level=0):
        indent = "  " * level
        choices_str_list = []
        for i, choice in enumerate(self.choices):
            choices_str_list.append(f"{indent}  [Choice {i+1}]:\n{choice._repr_indent(level + 2)}")
        return f"{indent}[ChoiceAction]({self.level_name})\n" + "\n".join(choices_str_list)

    def to_string_representation(self) -> str:
        return " ++ ".join([choice.to_string_representation() for choice in self.choices])

class SequenceActionConfig(ActionConfig):
    def __init__(self, actions: Optional[List[ActionConfig]] = None):
        self.actions = actions if actions is not None else []

    def _repr_indent(self, level=0):
        indent = "  " * level
        actions_str = "\n".join([action._repr_indent(level + 1) for action in self.actions])
        return f"{indent}[SequenceAction]\n{actions_str or f'{indent}  (None)'}"

    def to_string_representation(self) -> str:
        return "; ".join([action.to_string_representation() for action in self.actions])

class DeterministicAssignmentActionConfig(ActionConfig):
    def __init__(self, variable: str, value: str):
        self.variable = variable
        self.value = value

    def _repr_indent(self, level=0):
        indent = "  " * level
        return f"{indent}[DeterministicAssignment]: {self.variable} := '{self.value}'"

    def to_string_representation(self) -> str:
        return f"{self.variable} := {self.value}"

class NondeterministicAssignmentActionConfig(ActionConfig):
    def __init__(self, variable: str):
        self.variable = variable

    def _repr_indent(self, level=0):
        indent = "  " * level
        return f"{indent}[NondeterministicAssignment]: {self.variable} := *"

    def to_string_representation(self) -> str:
        return f"{self.variable} := *"

class ActionConstraintConfig(ActionConfig):
    def __init__(self, formula: str):
        self.formula = formula

    def _repr_indent(self, level=0):
        indent = "  " * level
        return f"{indent}[ActionConstraint]: '{self.formula}'"

    def to_string_representation(self) -> str:
        return f"?{self.formula}"

class CallHybridProgramConfig(ActionConfig):
    def __init__(self, hp_name: str):
        self.hp_name = hp_name

    def _repr_indent(self, level=0):
        indent = "  " * level
        return f"{indent}[CallHybridProgram]: {self.hp_name}"

    def to_string_representation(self) -> str:
        return self.hp_name

# Define ActionType for type hinting
ActionType = Union[
    ChoiceActionConfig,
    SequenceActionConfig,
    DeterministicAssignmentActionConfig,
    NondeterministicAssignmentActionConfig,
    ActionConstraintConfig,
    CallHybridProgramConfig,
    HybridProgramStepConfig # Added for loop body
]

# End of pyast_config.py content

import numpy as np
import math
import re
from typing import Dict, Any, Optional, Union, List, Tuple
from scipy.integrate import solve_ivp
import z3
import gymnasium as gym
from gymnasium import spaces

# Import all necessary classes from pyast_config
import pyast_config

class GeneratedKXEnv(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 30}

    def __init__(self, render_mode=None, cisb_size=100, cisb_populate_attempts=1000, seed=None):
        super().__init__()
        self.seed = seed
        self.np_random = np.random.default_rng(seed)

        self.render_mode = render_mode
        self.equality_tolerance = 1e-6 # Tolerance for floating point comparisons

        # --- Initialize ProblemConfig using pyast_config classes ---
        initial_precondition_formula = 'x=0 & y=0 & v=0 & dx=1 & dy=0 & t=0 & r=1 & xo=5 & yo=0 & ep=1 & b=1 & A=0 & W=0'
        postcondition_formula = 'x=xo & y=yo'
        loop_invariant_formula = 'isPhysicallyValid() & isWellformedDir() & bounds() & ensure()' # Example invariant

        self._problem_config = pyast_config.ProblemConfig(
            precondition=pyast_config.PreconditionConfig(formula=initial_precondition_formula),
            loop_execution=pyast_config.LoopExecutionConfig(
                loop_invariant_formula=loop_invariant_formula
            ),
            postcondition=pyast_config.PostconditionConfig(formula=postcondition_formula)
        )
        print(f"DEBUG: GeneratedKXEnv.__init__ - Initializing self._problem_config.precondition.formula to: '{self._problem_config.precondition.formula}'")

        # --- Initialize VariablesConfig using pyast_config classes ---
        self._variables_config = pyast_config.VariablesConfig(
            state_variables=[
                pyast_config.VariableDeclarationConfig('x', 'real'),
                pyast_config.VariableDeclarationConfig('y', 'real'),
                pyast_config.VariableDeclarationConfig('v', 'real'),
                pyast_config.VariableDeclarationConfig('a', 'real'),
                pyast_config.VariableDeclarationConfig('dx', 'real'),
                pyast_config.VariableDeclarationConfig('dy', 'real'),
                pyast_config.VariableDeclarationConfig('w', 'real'),
                pyast_config.VariableDeclarationConfig('t', 'real')
            ]
        )
        self._state_keys = [v.name for v in self._variables_config.state_variables]

        # --- Initialize DefinitionsConfig using pyast_config classes ---
        self._definitions_config = pyast_config.DefinitionsConfig(
            constants=[
                pyast_config.ConstantDefinitionConfig('r', 'real', '1'),
                pyast_config.ConstantDefinitionConfig('xo', 'real', '5'),
                pyast_config.ConstantDefinitionConfig('yo', 'real', '0'),
                pyast_config.ConstantDefinitionConfig('ep', 'real', '1'),
                pyast_config.ConstantDefinitionConfig('b', 'real', '1'),
                pyast_config.ConstantDefinitionConfig('A', 'real', '0'),
                pyast_config.ConstantDefinitionConfig('W', 'real', '0')
            ],
            functions=[
                pyast_config.FunctionDefinitionConfig('stopDist', 'real', [('real', 'v')], 'v^2 / (2*A)')
            ],
            predicates=[
                pyast_config.PredicateDefinitionConfig('isPhysicallyValid', [], 'v >= 0', {'type': 'safety', 'penalty': '-100.0', 'on_failure': 'terminate:status=failure'}),
                pyast_config.PredicateDefinitionConfig('isWellformedDir', [], 'dx^2 + dy^2 = 1', {'type': 'progress', 'shape': '-((dx^2 + dy^2) - 1)^2'}),
                pyast_config.PredicateDefinitionConfig('isEfficient', [], 'v > 0', {'type': 'progress', 'shape': 'v > 0 ? 1/v : v'}),
                pyast_config.PredicateDefinitionConfig('bounds', [], 'r > 0 & ep > 0 & b > 0 & A >= 0 & W >= 0', {'type': 'safety', 'penalty': '-200.0', 'on_failure': 'terminate:status=failure'}),
                pyast_config.PredicateDefinitionConfig('ensure', [], self._problem_config.postcondition.formula, {'type': 'goal', 'reward': '1000.0', 'on_success': 'terminate:status=success', 'penalty': '-500.0', 'on_failure': 'terminate:status=failure', 'shape': '-( (x-xo)^2 + (y-yo)^2 )'})
            ]
        )
        self.constants = {c.name: float(c.value) if c.value else 0.0 for c in self._definitions_config.constants}
        self.functions = {} # Will be populated with callable lambdas
        self.predicates = {} # Will be populated with callable lambdas

        self._compile_functions_and_predicates()

        self.state = {var.name: 0.0 for var in self._variables_config.state_variables}

        # Define observation space dynamically
        low = np.array([-np.inf] * len(self._state_keys), dtype=np.float32)
        high = np.array([np.inf] * len(self._state_keys), dtype=np.float32)
        self.observation_space = spaces.Box(low, high, dtype=np.float32)

        # --- Define action space using pyast_config classes ---
        self._control_func_action_config = pyast_config.ChoiceActionConfig(
            level_name="main_choice",
            choices=[
                pyast_config.SequenceActionConfig(actions=[
                    pyast_config.DeterministicAssignmentActionConfig('a', '1.0'),
                    pyast_config.DeterministicAssignmentActionConfig('dx', '0.0'),
                    pyast_config.DeterministicAssignmentActionConfig('dy', '1.0')
                ]),
                pyast_config.SequenceActionConfig(actions=[
                    pyast_config.DeterministicAssignmentActionConfig('a', '0.0'),
                    pyast_config.DeterministicAssignmentActionConfig('dx', '1.0'),
                    pyast_config.DeterministicAssignmentActionConfig('dy', '0.0')
                ])
            ]
        )
        self.action_space = self._get_action_space_from_config(self._control_func_action_config)

        self.max_episode_steps = 100 # Example max steps

        # ODE setup
        self._ode_state_map = {
            'x': 0, 'y': 1, 'v': 2, 'a': 3, 'dx': 4, 'dy': 5, 'w': 6, 't': 7
        }
        self._ode_func_compiled = self._compile_ode_function()
        self._ode_events_compiled = self._compile_ode_events()

        self._cisb = []
        self._cisb_size = cisb_size
        self._cisb_populate_attempts = cisb_populate_attempts # This is now mostly for debug output
        self._populate_cisb()

        self.last_failing_precondition_formula = None

    def _compile_functions_and_predicates(self):
        """Compiles KeYmaera X functions and predicates into callable Python lambdas."""
        # Functions
        for func_decl in self._definitions_config.functions:
            # Extract only names from (type, name) tuples
            param_names = [name for _, name in func_decl.parameters]
            # Ensure the body is pre-processed for identifiers before compiling
            processed_body = self._preprocess_kx_expression(func_decl.body)
            try:
                # Create a lambda function, ensuring 'self' is available for evaluation context
                # This requires a bit of trickery to pass 'self' into the lambda's scope
                # A more robust solution might involve a dedicated expression parser.
                # For now, we'll assume the body is simple enough to be evaluated with self.state/self.constants
                # and local parameters.
                # The _evaluate_expression method handles the actual evaluation.
                self.functions[func_decl.name] = lambda *args, _func_body=processed_body, _param_names=param_names: \
                    self._evaluate_expression(_func_body, local_vars=dict(zip(_param_names, args)))
            except Exception as e:
                print(f"Error compiling function '{func_decl.name}': {e}")
                self.functions[func_decl.name] = lambda *args: 0.0 # Fallback

        # Predicates
        for pred_decl in self._definitions_config.predicates:
            # Extract only names from (type, name) tuples
            param_names = [name for _, name in pred_decl.parameters]
            processed_body = self._preprocess_kx_expression(pred_decl.body)
            try:
                self.predicates[pred_decl.name] = lambda *args, _pred_body=processed_body, _param_names=param_names: \
                    self._evaluate_expression(_pred_body, local_vars=dict(zip(_param_names, args)), is_constraint=True)
            except Exception as e:
                print(f"Error compiling predicate '{pred_decl.name}': {e}")
                self.predicates[pred_decl.name] = lambda *args: False # Fallback

    def _preprocess_kx_expression(self, expr_str: str) -> str:
        """
        Pre-processes a KeYmaera X expression string to replace identifiers
        with their Python-friendly references (e.g., self.state['x']).
        This is a simplified version of the logic in _evaluate_expression
        to be used for pre-compilation.
        """
        processed_expr = expr_str.strip()

        all_env_identifiers = list(self._state_keys) + \
                              list(self.constants.keys()) + \
                              list(self.functions.keys()) + \
                              list(self.predicates.keys())

        # Sort by length descending to handle cases where one identifier is a substring of another
        all_env_identifiers_sorted = sorted(all_env_identifiers, key=len, reverse=True)

        for ident_name in all_env_identifiers_sorted:
            replacement = ""
            if ident_name in self._state_keys:
                replacement = f"self.state['{ident_name}']"
            elif ident_name in self.constants:
                replacement = f"self.constants['{ident_name}']"
            elif ident_name in self.functions:
                # For functions, we need to call them, so just the name is fine,
                # as the lambda itself will be in self.functions.
                # The actual call will be handled by eval.
                replacement = f"self.functions['{ident_name}']"
            elif ident_name in self.predicates:
                replacement = f"self.predicates['{ident_name}']"

            if replacement:
                # Use regex with word boundaries to ensure we replace whole identifiers only
                # This replacement is for the *body* of the function/predicate, not its call site.
                # The _evaluate_expression handles the actual call site logic.
                processed_expr = re.sub(r'\b' + re.escape(ident_name) + r'\b', replacement, processed_expr)
        return processed_expr

    def _get_predicate_metadata(self, predicate_name: str) -> Dict[str, Any]:
        """Retrieves metadata for a given predicate name."""
        for pred_decl in self._definitions_config.predicates:
            if pred_decl.name == predicate_name:
                return pred_decl.metadata
        # Default metadata if not found
        return {'type': 'safety', 'penalty': '-100.0', 'on_failure': 'terminate:status=failure'}

    def _get_action_space_from_config(self, config: pyast_config.ActionType) -> spaces.Space:
        """Dynamically creates the action space based on the control function configuration."""
        if isinstance(config, pyast_config.ChoiceActionConfig):
            # A choice action maps to a Discrete space
            return spaces.Discrete(len(config.choices))
        elif isinstance(config, pyast_config.SequenceActionConfig):
            # If the top-level is a sequence, it implies no direct agent choice,
            # but rather a sequence of deterministic or nondeterministic actions.
            # We need to find the first actual choice or nondet assignment.
            for sub_action in config.actions:
                if isinstance(sub_action, pyast_config.ChoiceActionConfig):
                    return self._get_action_space_from_config(sub_action)
                elif isinstance(sub_action, pyast_config.NondeterministicAssignmentActionConfig):
                    # For non-deterministic assignments, we need to define a continuous space
                    # for the agent to choose the value.
                    # This is a simplification; a real implementation might need bounds or more info.
                    return spaces.Box(low=-10.0, high=10.0, shape=(1,), dtype=np.float32)
            # If no choices or nondet assignments, it's a fixed sequence, so a no-op action space.
            return spaces.Discrete(1) # Represents a single 'no-op' action
        elif isinstance(config, pyast_config.NondeterministicAssignmentActionConfig):
            # Direct non-deterministic assignment
            return spaces.Box(low=-10.0, high=10.0, shape=(1,), dtype=np.float32)
        else:
            # For deterministic assignments or constraints, there's no agent choice at this level.
            return spaces.Discrete(1) # Represents a single 'no-op' action

    def _get_obs(self):
        """Returns the current observation of the environment."""
        return np.array([self.state[key] for key in self._state_keys], dtype=np.float32)

    def _get_info(self):
        """Returns auxiliary information."""
        return {"current_step": self.current_step}

    def _compile_ode_function(self):
        """
        Compiles the ODE system into a callable function for scipy.integrate.solve_ivp.
        The ODEs are hardcoded based on the problem description.
        """
        def ode_func(t, y):
            # Map y values back to state variable names
            state_vals = {name: y[idx] for name, idx in self._ode_state_map.items()}

            # Retrieve constants and current 'a' value from self.state
            current_a = state_vals['a']
            current_w = state_vals['w'] # Assuming 'w' is a state variable that can change

            # Differential equations (hardcoded for now based on common robot car model)
            # x' = v * dx
            # y' = v * dy
            # v' = a (acceleration)
            # dx' = -w * dy (angular velocity changes direction)
            # dy' = w * dx
            # w' = 0 (assuming w is set by control and constant during ODE phase)
            # t' = 1

            # Retrieve current values
            x, y, v, a, dx, dy, w, t = y[0], y[1], y[2], y[3], y[4], y[5], y[6], y[7]

            dydt = np.zeros_like(y)
            dydt[self._ode_state_map['x']] = v * dx
            dydt[self._ode_state_map['y']] = v * dy
            dydt[self._ode_state_map['v']] = a
            dydt[self._ode_state_map['a']] = 0.0 # 'a' is set by control, constant during ODE
            dydt[self._ode_state_map['dx']] = -w * dy
            dydt[self._ode_state_map['dy']] = w * dx
            dydt[self._ode_state_map['w']] = 0.0 # 'w' is set by control, constant during ODE
            dydt[self._ode_state_map['t']] = 1.0

            return dydt
        return ode_func

    def _compile_ode_events(self):
        """
        Compiles ODE events for scipy.integrate.solve_ivp.
        Events include reaching 'ep' time, or v becoming negative.
        """
        # Event 1: Time reaches 'ep'
        def time_event(t, y):
            return y[self._ode_state_map['t']] - self.constants['ep']
        time_event.terminal = True
        time_event.direction = 1 # Detect when value crosses zero from negative to positive

        # Event 2: Velocity 'v' becomes negative (physical constraint violation)
        def v_negative_event(t, y):
            return y[self._ode_state_map['v']] # Detect when v crosses zero
        v_negative_event.terminal = True
        v_negative_event.direction = -1 # Detect when value crosses zero from positive to negative

        return [time_event, v_negative_event]

    def _expand_formula(self, formula: str, definitions: pyast_config.DefinitionsConfig) -> str:
        """
        Expands functions and predicates within a formula string.
        This is a recursive process to handle nested definitions.
        """
        expanded_formula = formula

        # Combine functions and predicates for expansion
        expandables = {}
        for f in definitions.functions:
            expandables[f.name] = (f.body, [name for _, name in f.parameters]) # Extract only names
        for p in definitions.predicates:
            expandables[p.name] = (p.body, [name for _, name in p.parameters]) # Extract only names

        max_iterations = 10 # Prevent infinite loops for pathological recursive definitions
        for _ in range(max_iterations):
            num_substitutions_in_pass = 0
            # Sort by name length descending to avoid issues where one identifier is a substring of another
            sorted_names = sorted(expandables.keys(), key=len, reverse=True)

            for name in sorted_names:
                body, params = expandables[name]
                # Regex to find calls to the function/predicate, e.g., "func(arg1, arg2)"
                # \b ensures whole word match, \s* allows for whitespace
                # ([^)]*) captures arguments inside parentheses
                pattern = r'\b' + re.escape(name) + r'\s*\(([^)]*)\)'

                def replace_match(match):
                    nonlocal num_substitutions_in_pass
                    num_substitutions_in_pass += 1
                    call_args_str = match.group(1)
                    # Split arguments, handling commas inside nested parentheses
                    call_args = []
                    current_arg = []
                    paren_level = 0
                    for char in call_args_str:
                        if char == '(':
                            paren_level += 1
                        elif char == ')':
                            paren_level -= 1
                        elif char == ',' and paren_level == 0:
                            call_args.append("".join(current_arg).strip())
                            current_arg = []
                            continue
                        current_arg.append(char)
                    if current_arg:
                        call_args.append("".join(current_arg).strip())

                    if len(call_args) != len(params):
                        # Mismatch in arguments, return original call to indicate error or unhandled case
                        print(f"Warning: Argument count mismatch for {name}. Expected {len(params)}, got {len(call_args)} in call '{match.group(0)}'")
                        return match.group(0)

                    substituted_body = body
                    for i, param_name in enumerate(params):
                        # Use word boundaries to avoid replacing parts of other identifiers
                        substituted_body = re.sub(r'\b' + re.escape(param_name) + r'\b', f"({call_args[i]})", substituted_body)
                    return f"({substituted_body})" # Enclose substituted body in parentheses for correct precedence

                # Perform replacement
                new_expanded_formula, count = re.subn(pattern, replace_match, expanded_formula)
                expanded_formula = new_expanded_formula
                # num_substitutions_in_pass is updated by the nested function

            if num_substitutions_in_pass == 0:
                break # No more substitutions were made in this pass, so we are done
        return expanded_formula

    def _apply_meta_effect(self, metadata: Dict[str, Any], current_reward: float, current_terminated: bool, current_truncated: bool, is_failure_case: bool) -> Tuple[float, bool, bool]:
        """
        Applies reward, termination, or truncation effects based on metadata.
        """
        reward_change = 0.0
        new_terminated = current_terminated
        new_truncated = current_truncated

        if metadata:
            event_key = 'on_failure' if is_failure_case else 'on_success'
            event_config_str = metadata.get(event_key)

            if is_failure_case:
                penalty = float(metadata.get('penalty', 0.0))
                reward_change += penalty
            else: # Success case
                success_reward = float(metadata.get('reward', 0.0))
                reward_change += success_reward

            if event_config_str:
                parts = event_config_str.split(':')
                action = parts[0]
                # status_part = parts[1] if len(parts) > 1 else 'status=none'
                # status = status_part.split('=')[1] # Not directly used for logic, but could be for info

                if action == 'terminate':
                    new_terminated = True
                elif action == 'truncate':
                    new_truncated = True
                # 'continue' means no change to terminated/truncated state, so no 'else' needed here.

        return current_reward + reward_change, new_terminated, new_truncated

    def _evaluate_shape_expression(self, shape_expr: str) -> float:
        """
        Evaluates a shape expression from metadata.
        Shape expressions might use ternary operators (e.g., 'v > 0 ? 1/v : v').
        """
        if not shape_expr:
            return 0.0

        # Handle ternary operator 'condition ? value_if_true : value_if_false'
        # This is a simple regex-based parser, might need more robustness for complex cases.
        match = re.match(r'(.+)\s*\?\s*(.+)\s*:\s*(.+)', shape_expr)
        if match:
            condition_str, true_val_str, false_val_str = match.groups()
            try:
                condition_result = self._evaluate_expression(condition_str)
                if condition_result:
                    return float(self._evaluate_expression(true_val_str))
                else:
                    return float(self._evaluate_expression(false_val_str))
            except Exception as e:
                print(f"Warning: Error evaluating shape ternary expression '{shape_expr}': {e}")
                return 0.0
        else:
            # If not a ternary, just evaluate as a regular expression
            try:
                return float(self._evaluate_expression(shape_expr))
            except Exception as e:
                print(f"Warning: Error evaluating shape expression '{shape_expr}': {e}")
                return 0.0


    def _kx_expr_to_z3_formula(self, kx_expr: str, z3_symbol_map: Dict[str, Any], fixed_constants: Dict[str, Union[int, float]]) -> Any:
        """
        Converts a KeYmaera X expression string into a Z3 formula.
        This function performs string replacements to convert KeYmaera X syntax
        to a Python string that, when evaluated, constructs a Z3 expression.
        It processes operators respecting precedence: arithmetic, relational, then logical.
        For logical operators, it uses a recursive approach to handle nested parentheses.
        """
        print(f"DEBUG: _kx_expr_to_z3_formula - Initial kx_expr: '{kx_expr}'")
        expr_transformed = kx_expr.strip()
        print(f"DEBUG: _kx_expr_to_z3_formula - After strip: '{expr_transformed}'")
        if not expr_transformed:
            raise ValueError(f"Internal error: kx_expr became empty after stripping. Original: '{kx_expr}'")


        # A helper for expanding functions/predicates for Z3
        def _expand_for_z3(formula_to_expand: str, definitions_config: pyast_config.DefinitionsConfig) -> str:
            expanded = formula_to_expand
            # Map of name -> (body, params) for both functions and predicates
            expandables = {}
            for f in definitions_config.functions:
                expandables[f.name] = (f.body, [name for _, name in f.parameters]) # Extract only names
            for p in definitions_config.predicates:
                expandables[p.name] = (p.body, [name for _, name in p.parameters]) # Extract only names

            while True:
                num_substitutions_in_pass = 0
                sorted_names = sorted(expandables.keys(), key=len, reverse=True)

                for name in sorted_names:
                    body, params = expandables[name]
                    pattern = r'\b' + re.escape(name) + r'\s*\(([^)]*)\)'

                    def replace_match_z3(match):
                        nonlocal num_substitutions_in_pass
                        num_substitutions_in_pass += 1
                        call_args_str = match.group(1)
                        call_args = [arg.strip() for arg in call_args_str.split(',') if arg.strip()]

                        if len(call_args) != len(params):
                            # Mismatch in args, return original call for Z3 to potentially fail later
                            return match.group(0)

                        substituted_body = body
                        for i, param_name in enumerate(params):
                            substituted_body = re.sub(r'\b' + re.escape(param_name) + r'\b', call_args[i], substituted_body)
                        return f"({substituted_body})" # Enclose in parens

                    expanded, count = re.subn(pattern, replace_match_z3, expanded)
                if num_substitutions_in_pass == 0:
                    break
            return expanded

        # First, expand all function and predicate calls within the expression
        combined_definitions = pyast_config.DefinitionsConfig()
        combined_definitions.functions = self._definitions_config.functions
        combined_definitions.predicates = self._definitions_config.predicates

        expr_transformed = _expand_for_z3(expr_transformed, combined_definitions)
        print(f"DEBUG: _kx_expr_to_z3_formula - After _expand_for_z3: '{expr_transformed}'")
        if not expr_transformed.strip(): # Check if it's empty or just whitespace
            raise ValueError(f"Internal error: expr_transformed became empty or only whitespace after _expand_for_z3. Original kx_expr: '{kx_expr}'")


        # 1. Replace identifiers with their Z3 symbol map reference.
        all_known_kx_identifiers = list(z3_symbol_map.keys()) + \
                                   list(fixed_constants.keys())

        sorted_identifiers = sorted(all_known_kx_identifiers, key=len, reverse=True)

        for ident_name in sorted_identifiers:
            replacement = ""
            if ident_name in z3_symbol_map:
                replacement = f"z3_symbol_map['{ident_name}']"
            elif ident_name in fixed_constants:
                replacement = ident_name # Will be available in eval_globals

            if replacement:
                # Use word boundaries to avoid partial replacements (e.g., 'x' in 'dx')
                expr_transformed = re.sub(r'\b' + re.escape(ident_name) + r'\b', replacement, expr_transformed)

        print(f"DEBUG: _kx_expr_to_z3_formula - After identifier replacement: '{expr_transformed}'")
        if not expr_transformed.strip(): # Check if it's empty or just whitespace
            raise ValueError(f"Internal error: expr_transformed became empty or only whitespace after identifier replacement. Original kx_expr: '{kx_expr}'")


        # 2. Basic arithmetic and math functions
        expr_transformed = expr_transformed.replace('^', '**')
        expr_transformed = expr_transformed.replace('abs(', 'z3.Abs(') # Use z3.Abs for Z3 context
        print(f"DEBUG: _kx_expr_to_z3_formula - After arithmetic/math replacements: '{expr_transformed}'")
        if not expr_transformed.strip(): # Check if it's empty or just whitespace
            raise ValueError(f"Internal error: expr_transformed became empty or only whitespace after arithmetic/math replacements. Original kx_expr: '{kx_expr}'")


        # 3. Handle logical and relational operators respecting precedence and parentheses
        def _parse_logical_and_relational_expression_z3(expression_str_to_parse: str) -> str:
            # print(f"DEBUG: _parse_logical_and_relational_expression_z3 input: '{expression_str_to_parse}'")
            expression_str_to_parse = expression_str_to_parse.strip()
            if not expression_str_to_parse:
                # print("DEBUG: _parse_logical_and_relational_expression_z3 returning empty string (input was empty)")
                return ""

            # 1. Strip outermost balanced parentheses
            while expression_str_to_parse.startswith('(') and expression_str_to_parse.endswith(')'):
                balance = 0
                is_fully_enclosed = True
                for i in range(len(expression_str_to_parse) - 1): # Check up to second to last char
                    char = expression_str_to_parse[i]
                    if char == '(': balance += 1
                    elif char == ')': balance -= 1
                    if balance == 0:
                        is_fully_enclosed = False
                        break
                if is_fully_enclosed:
                    expression_str_to_parse = expression_str_to_parse[1:-1].strip()
                else:
                    break

            # --- Logical Operators (highest level of this parser) ---

            # Implication (lowest logical precedence, right-associative)
            balance = 0
            implication_idx = -1
            for i in range(len(expression_str_to_parse) - 2, -1, -1): # Scan right-to-left for implication
                char = expression_str_to_parse[i]
                if char == ')': balance += 1
                elif char == '(': balance -= 1
                elif balance == 0 and expression_str_to_parse[i:i+2] == '->':
                    implication_idx = i
                    break
            if implication_idx != -1:
                left = _parse_logical_and_relational_expression_z3(expression_str_to_parse[:implication_idx].strip())
                right = _parse_logical_and_relational_expression_z3(expression_str_to_parse[implication_idx+2:].strip())
                return f"z3.Implies({left}, {right})"

            # Disjunction (medium precedence, left-associative)
            balance = 0
            disjunction_idx = -1
            for i in range(len(expression_str_to_parse)): # Scan left-to-right for disjunction
                char = expression_str_to_parse[i]
                if char == '(': balance += 1
                elif char == ')': balance -= 1
                elif balance == 0 and char == '|':
                    disjunction_idx = i
                    break
            if disjunction_idx != -1:
                left = _parse_logical_and_relational_expression_z3(expression_str_to_parse[:disjunction_idx].strip())
                right = _parse_logical_and_relational_expression_z3(expression_str_to_parse[disjunction_idx+1:].strip())
                return f"z3.Or({left}, {right})"

            # Conjunction (highest precedence, left-associative)
            balance = 0
            conjunction_idx = -1
            for i in range(len(expression_str_to_parse)): # Scan left-to-right for conjunction
                char = expression_str_to_parse[i]
                if char == '(': balance += 1
                elif char == ')': balance -= 1
                elif balance == 0 and char == '&':
                    conjunction_idx = i
                    break
            if conjunction_idx != -1:
                left = _parse_logical_and_relational_expression_z3(expression_str_to_parse[:conjunction_idx].strip())
                right = _parse_logical_and_relational_expression_z3(expression_str_to_parse[conjunction_idx+1:].strip())
                return f"z3.And({left}, {right})"

            # Logical NOT (prefix operator)
            if expression_str_to_parse.startswith('!'):
                operand = _parse_logical_and_relational_expression_z3(expression_str_to_parse[1:].strip())
                return f"z3.Not({operand})"

            # --- Relational Operators (next level down) ---
            # Note: '=' in KeYmaera X maps to '==' for Z3 exact equality
            relational_ops = [('!=', '!='), ('<=', '<='), ('>=', '>='), ('<', '<'), ('>', '>'), ('=', '==')]

            balance = 0
            rel_op_found_kx = None
            rel_op_found_py = None
            rel_op_idx = -1

            # Scan left-to-right for relational operators
            for i in range(len(expression_str_to_parse)):
                char = expression_str_to_parse[i]
                if char == '(': balance += 1
                elif char == ')': balance -= 1
                elif balance == 0:
                    # Check for multi-character operators first
                    for kx_op, py_op_format in relational_ops:
                        if len(kx_op) > 1 and expression_str_to_parse[i:i+len(kx_op)] == kx_op:
                            rel_op_found_kx = kx_op
                            rel_op_found_py = py_op_format
                            rel_op_idx = i
                            break
                    if rel_op_found_kx: break

                    # Check for single-character operators if no multi-char op found yet
                    for kx_op, py_op_format in relational_ops:
                        if len(kx_op) == 1 and expression_str_to_parse[i] == kx_op:
                            # Special check for '=', ensure it's not part of '==', '!=', '<=', '>='
                            if kx_op == '=' and ((i > 0 and expression_str_to_parse[i-1] in '<>=!') or \
                                            (i < len(expression_str_to_parse) - 1 and expression_str_to_parse[i+1] == '=')):
                                continue
                            rel_op_found_kx = kx_op
                            rel_op_found_py = py_op_format
                            rel_op_idx = i
                            break
                    if rel_op_found_kx: break

            if rel_op_found_kx:
                left = _parse_logical_and_relational_expression_z3(expression_str_to_parse[:rel_op_idx].strip())
                right = _parse_logical_and_relational_expression_z3(expression_str_to_parse[rel_op_idx+len(rel_op_found_kx):].strip())
                # For Z3, '==' is used for equality, and other relational ops are direct
                return f"({left}) {rel_op_found_py} ({right})"


            # Base case: no logical or relational operators found, return the expression as is.
            # This should now be an arithmetic expression or a literal.
            return expression_str_to_parse

        parsed_expr_transformed = _parse_logical_and_relational_expression_z3(expr_transformed)
        print(f"DEBUG: _kx_expr_to_z3_formula - After _parse_logical_and_relational_expression_z3: '{parsed_expr_transformed}'")
        if not parsed_expr_transformed.strip():
            raise ValueError(f"Internal error: parsed_expr_transformed became empty or only whitespace after logical/relational parsing. Original kx_expr: '{kx_expr}'")

        expr_transformed = parsed_expr_transformed # Assign the parsed result back

        # Final evaluation in a Z3-aware context
        eval_globals = {'z3': z3, 'z3_symbol_map': z3_symbol_map}
        for const_name, const_value in fixed_constants.items():
            if isinstance(const_value, int):
                eval_globals[const_name] = z3.IntVal(const_value)
            else: # float or other numeric
                eval_globals[const_name] = z3.RealVal(const_value)

        try:
            return eval(expr_transformed, eval_globals)
        except Exception as e:
            raise ValueError(f"Failed to evaluate Z3 expression '{kx_expr}' (transformed: '{expr_transformed}'): {e}")


    def _solve_initial_state_with_z3(self, precondition_kx: str, hint_values: Optional[Dict[str, float]] = None) -> Optional[Dict[str, float]]:
        """
        Uses Z3 to find a satisfying assignment for initial state variables and constants
        based on the expanded KeYmaera X precondition, guided by optional hint_values.
        Returns a dictionary of solved values (variable/constant name -> float) or None if unsatisfiable.
        """
        print(f"DEBUG: _solve_initial_state_with_z3 - Received precondition_kx: '{precondition_kx}'")

        # 1. Prepare Z3 symbols (Real for all variables and constants that need to be solved)
        z3_symbol_map = {}

        # Store fixed constants separately to avoid creating Z3 variables for them
        fixed_constants = {}

        # State variables (these are the ones we want Z3 to solve for)
        for var_decl in self._variables_config.state_variables:
            z3_symbol_map[var_decl.name] = z3.Real(var_decl.name)

        # Constants: If they have a fixed numeric value, use that value directly in the expression.
        # Otherwise, treat them as Z3 Real variables to be solved for.
        for const_decl in self._definitions_config.constants:
            if const_decl.value is not None and re.fullmatch(r'^-?\d+(\.\d+)?$', const_decl.value):
                # This constant has a fixed numeric value. Store it as a Python float.
                fixed_constants[const_decl.name] = float(const_decl.value)
            else:
                # This constant either has no value, or its value is an expression.
                # For Z3, we treat it as a variable that needs to be solved.
                z3_symbol_map[const_decl.name] = z3.Real(const_decl.name)

        # 2. Convert expanded KeYmaera X formula to Z3 expression
        try:
            # Pass original precondition, expansion happens inside _kx_expr_to_z3_formula
            z3_formula = self._kx_expr_to_z3_formula(precondition_kx, z3_symbol_map, fixed_constants)
        except ValueError as e:
            print(f"Z3 Solver Error: Failed to convert precondition to Z3 formula: {e}")
            return None

        # 3. Create Z3 solver and add the formula
        s = z3.Solver()
        s.add(z3_formula)

        # 4. Add constraints based on hint_values to guide Z3 to diverse solutions
        # This block is skipped if hint_values is None, which is desired for initial CISB population
        if hint_values:
            epsilon_dir = 1e-6 # Tight tolerance for dx, dy
            epsilon_pos_r = 0.5 # Looser tolerance for positions and radius, constants

            for var_name, sampled_value in hint_values.items():
                if var_name in z3_symbol_map: # Only add hints for variables Z3 is solving for
                    if var_name == 'v':
                        s.add(z3_symbol_map[var_name] == 0.0) # v is strictly 0 by precondition
                    elif var_name == 't':
                        s.add(z3_symbol_map[var_name] == 0.0) # t is strictly 0 by problem setup
                    elif var_name in ['dx', 'dy']:
                        # Guide dx, dy close to sampled cos/sin values
                        s.add(z3_symbol_map[var_name] >= sampled_value - epsilon_dir)
                        s.add(z3_symbol_map[var_name] <= sampled_value + epsilon_dir)
                    else: # For x, y, xo, yo, r, ep, b, A, W
                        s.add(z3_symbol_map[var_name] >= sampled_value - epsilon_pos_r)
                        s.add(z3_symbol_map[var_name] <= sampled_value + epsilon_pos_r)
                        # Reinforce hard constraints that might be part of the precondition's expanded form
                        # These are already in z3_formula, but adding them again doesn't hurt and might
                        # make the solver's job easier if the hints are "far"
                        if var_name == 'r':
                            s.add(z3_symbol_map[var_name] != 0) # From initialState
                        if var_name == 'b':
                            s.add(z3_symbol_map[var_name] > 0) # From bounds
                        if var_name == 'ep':
                            s.add(z3_symbol_map[var_name] > 0) # From bounds
                        if var_name == 'A':
                            s.add(z3_symbol_map[var_name] >= 0) # From bounds


        # 5. Check for satisfiability
        check_result = s.check()
        if check_result == z3.sat:
            model = s.model()
            solved_values = {}
            # Add values for state variables and constants that were Z3 variables
            for k, v in z3_symbol_map.items():
                evaluated_val = model.evaluate(v, model_completion=True)
                if evaluated_val is not None:
                    # Corrected logic for converting Z3 values to Python floats
                    if z3.is_real(evaluated_val):
                        # as_decimal is preferred for direct float conversion if available and precise enough
                        try:
                            solved_values[k] = float(evaluated_val.as_decimal(15))
                        except AttributeError: # as_decimal might not be available on all Real types, fallback to as_fraction
                            solved_values[k] = float(evaluated_val.as_fraction())
                    elif z3.is_int(evaluated_val):
                        # For Z3 Int values
                        solved_values[k] = float(evaluated_val.as_long())
                    else:
                        # Fallback for other Z3 types or if specific methods are not present
                        try:
                            solved_values[k] = float(str(evaluated_val)) # Convert to string and then float
                        except ValueError as e:
                            print(f"Error: Could not convert Z3 value '{evaluated_val}' for '{k}' to float: {e}")
                            solved_values[k] = 0.0 # Default to 0.0 on conversion failure
            # Add values for fixed constants
            # These are already numeric, no need for Z3 evaluation
            solved_values.update(fixed_constants)
            return solved_values
        elif check_result == z3.unsat:
            # print(f"Z3 UNSAT for precondition: {precondition_kx} with hints {hint_values}") # For debugging Z3 unsatisfiability
            # print(f"  Solver constraints: {s.assertions()}")
            return None
        else: # z3.unknown
            print(f"Z3 UNKNOWN for precondition: {precondition_kx} with hints {hint_values}")
            return None


    def _evaluate_expression(self, expr_str: str, local_vars: Optional[Dict[str, Any]] = None, is_constraint: bool = False) -> Any:
        """
        Safe evaluation of expressions using environment's context.
        This method parses KeYmaera X expressions at runtime.
        If is_constraint is True, returns False on evaluation error.
        """

        if local_vars is None:
            local_vars = {}

        # Strip KeYmaera X constraint prefix '?' if present
        if expr_str.startswith('?'):
            expr_str = expr_str[1:].strip()

        # --- NEW PRE-PROCESSING STEP: Prefix identifiers ---
        # This step converts bare KeYmaera X identifiers (like 'x', 'A', 'stopDist')
        # into Python-compatible references (like 'self.state["x"]', 'self.constants["A"]', 'self.functions["stopDist"]')
        # before the expression is further parsed and evaluated.

        processed_expr = expr_str.strip()

        # Combine all known identifiers from the environment that need prefixing
        # Exclude 'np' and 'math' as they are global modules and should not be prefixed.
        all_env_identifiers = list(self.state.keys()) + \
                              list(self.constants.keys()) + \
                              list(self.functions.keys()) + \
                              list(self.predicates.keys())

        # Sort by length descending to handle cases where one identifier is a substring of another (e.g., 'dx' before 'x')
        all_env_identifiers_sorted = sorted(all_env_identifiers, key=len, reverse=True)

        # Create a temporary list of identifiers that are *local* to this evaluation (e.g., lambda parameters passed in local_vars)
        # These should NOT be prefixed.
        local_identifiers_in_scope = list(local_vars.keys()) if local_vars else []

        # Perform the prefixing
        for ident_name in all_env_identifiers_sorted:
            if ident_name in local_identifiers_in_scope:
                continue # Skip if it's a local variable (e.g., 'v' in 'stopDist(v)' if 'v' is in local_vars)

            replacement = ""
            if ident_name in self.state:
                replacement = f"self.state['{ident_name}']"
            elif ident_name in self.constants:
                replacement = f"self.constants['{ident_name}']"
            elif ident_name in self.functions:
                replacement = f"self.functions['{ident_name}']"
            elif ident_name in self.predicates:
                replacement = f"self.predicates['{ident_name}']"

            if replacement:
                # Use regex with word boundaries to ensure we replace whole identifiers only
                processed_expr = re.sub(r'\b' + re.escape(ident_name) + r'\b', replacement, processed_expr)

        # --- END NEW PRE-PROCESSING STEP ---

        expr_py_transformed = processed_expr # Now use the processed_expr

        # 1. Basic syntax replacements (power, abs)
        expr_py_transformed = expr_py_transformed.replace('^', '**').replace('abs(', 'np.abs(')

        # 2. Recursive parsing for operator precedence (relational then logical)
        def _parse_expression_recursive_runtime(sub_expr: str) -> str:
            sub_expr = sub_expr.strip()
            if not sub_expr:
                return ""

            # Handle outermost balanced parentheses
            if sub_expr.startswith('(') and sub_expr.endswith(')'):
                balance = 0
                is_fully_enclosed = True
                for i in range(len(sub_expr) - 1): # Check up to second to last char
                    char = sub_expr[i]
                    if char == '(': balance += 1
                    elif char == ')': balance -= 1
                    if balance == 0:
                        is_fully_enclosed = False
                        break
                if is_fully_enclosed:
                    return _parse_expression_recursive_runtime(sub_expr[1:-1]) # Recursively parse inner part

            # Find lowest precedence operator at top level

            # Implication (lowest logical precedence) - right-associative
            balance = 0
            implication_idx = -1
            for i in range(len(sub_expr) - 2, -1, -1): # Scan right-to-left
                char_pair = sub_expr[i:i+2]
                if char_pair == '->' and balance == 0:
                    implication_idx = i
                    break
                if sub_expr[i] == ')': balance += 1
                elif sub_expr[i] == '(': balance -= 1

            if implication_idx != -1:
                left = _parse_expression_recursive_runtime(sub_expr[:implication_idx].strip())
                right = _parse_expression_recursive_runtime(sub_expr[implication_idx+2:].strip())
                return f"((not ({left})) or ({right}))"

            # Disjunction (next lowest logical) - left-associative
            balance = 0
            disjunction_idx = -1
            for i in range(len(sub_expr)): # Scan left-to-right
                if sub_expr[i] == '|' and balance == 0:
                    disjunction_idx = i
                    break
                if sub_expr[i] == '(': balance += 1
                elif sub_expr[i] == ')': balance -= 1

            if disjunction_idx != -1:
                left = _parse_expression_recursive_runtime(sub_expr[:disjunction_idx].strip())
                right = _parse_expression_recursive_runtime(sub_expr[disjunction_idx+1:].strip())
                return f"({left}) or ({right})"

            # Conjunction (highest precedence, left-associative)
            balance = 0
            conjunction_idx = -1
            for i in range(len(sub_expr)): # Scan left-to-right
                if sub_expr[i] == '&' and balance == 0:
                    conjunction_idx = i
                    break
                if sub_expr[i] == '(': balance += 1
                elif sub_expr[i] == ')': balance -= 1

            if conjunction_idx != -1:
                left = _parse_expression_recursive_runtime(sub_expr[:conjunction_idx].strip())
                right = _parse_expression_recursive_runtime(sub_expr[conjunction_idx+1:].strip())
                return f"({left}) and ({right})"

            # Negation (highest logical precedence) - prefix operator
            if sub_expr.startswith('!'):
                operand = _parse_expression_recursive_runtime(sub_expr[1:].strip())
                return f"(not ({operand}))"

            # Relational operators (including '=')
            relational_ops = [('!=', '!='), ('<=', '<='), ('>=', '>='), ('<', '<'), ('>', '>'), ('=', '_approx_eq')]

            balance = 0
            rel_op_found_kx = None
            rel_op_found_py = None
            rel_op_idx = -1

            # Scan left-to-right for relational operators
            for i in range(len(sub_expr)):
                char = sub_expr[i]
                if char == '(': balance += 1
                elif char == ')': balance -= 1
                elif balance == 0:
                    # Check for multi-character operators first
                    for kx_op, py_op_format in relational_ops:
                        if len(kx_op) > 1 and sub_expr[i:i+len(kx_op)] == kx_op:
                            rel_op_found_kx = kx_op
                            rel_op_found_py = py_op_format
                            rel_op_idx = i
                            break
                    if rel_op_found_kx: break

                    # Check for single-character operators if no multi-char op found yet
                    for kx_op, py_op_format in relational_ops:
                        if len(kx_op) == 1 and sub_expr[i] == kx_op:
                            # Special check for '=', ensure it's not part of '==', '!=', '<=', '>='
                            if kx_op == '=' and ((i > 0 and sub_expr[i-1] in '<>=!') or \
                                            (i < len(sub_expr) - 1 and sub_expr[i+1] == '=')):
                                continue
                            rel_op_found_kx = kx_op
                            rel_op_found_py = py_op_format
                            rel_op_idx = i
                            break
                    if rel_op_found_kx: break

            if rel_op_found_kx:
                left = _parse_expression_recursive_runtime(sub_expr[:rel_op_idx].strip())
                right = _parse_expression_recursive_runtime(sub_expr[rel_op_idx+len(rel_op_found_kx):].strip())
                if rel_op_found_kx == '=':
                    return f"{rel_op_found_py}({left}, {right})"
                else:
                    return f"({left}) {rel_op_found_py} ({right})"

            # If no logical or relational operators, it's an arithmetic expression, literal, or function/predicate call
            return sub_expr

        expr_py_final = _parse_expression_recursive_runtime(expr_py_transformed)
        # --- End KeYmaera X to Python conversion logic ---

        try:
            # With explicit prefixing, eval_locals can be much simpler.
            # Only local_vars are truly 'local' to the eval call.
            # All other references are now explicit (self.state['x'], self.constants['A'], etc.)
            eval_globals_for_eval = {'np': np, 'math': math, '_approx_eq': self._approx_eq, 'self': self}
            result = eval(expr_py_final, eval_globals_for_eval, local_vars)
            return result
        except Exception as e:
            print(f"Error evaluating expression '{expr_str}' (transformed: '{expr_py_final}'): {e}")
            if is_constraint:
                return False # For constraints, return False on error
            raise # Re-raise for non-constraint expressions

    def _approx_eq(self, val1, val2) -> bool:
        """Helper for approximate equality with tolerance for floating point numbers."""
        return abs(val1 - val2) < self.equality_tolerance


    def _execute_action_config(self, action_config: pyast_config.ActionType, agent_action: Dict[str, Any]) -> Tuple[float, bool, bool]:
        """
        Recursively executes a given action configuration (Choice, Sequence, Assignment, Constraint).
        Returns (reward, terminated, applied_successfully).
        """
        reward = 0.0
        terminated = False
        applied_successfully = True

        if isinstance(action_config, pyast_config.ChoiceActionConfig):
            choice_key = f"choice_{action_config.level_name}"
            if choice_key not in agent_action:
                print(f"Error: Choice key '{choice_key}' missing from agent action.")
                reward -= 200.0
                terminated = True
                applied_successfully = False
                return reward, terminated, applied_successfully

            choice_idx = agent_action[choice_key]
            if not (0 <= choice_idx < len(action_config.choices)):
                print(f"Error: Invalid choice index {choice_idx} for '{choice_key}'.")
                reward -= 200.0
                terminated = True
                applied_successfully = False
                return reward, terminated, applied_successfully

            chosen_sub_config = action_config.choices[choice_idx]
            return self._execute_action_config(chosen_sub_config, agent_action)

        elif isinstance(action_config, pyast_config.SequenceActionConfig):
            for sub_action in action_config.actions:
                r, t, s = self._execute_action_config(sub_action, agent_action)
                reward += r
                if t: terminated = True
                if not s:
                    applied_successfully = False
                    break # Stop processing sequence if any sub-action fails
            return reward, terminated, applied_successfully

        elif isinstance(action_config, pyast_config.DeterministicAssignmentActionConfig):
            try:
                # Value is a raw KeYmaera X string, parsed at runtime by _evaluate_expression
                assigned_value = self._evaluate_expression(action_config.value)
                self.state[action_config.variable] = assigned_value
            except Exception as e:
                print(f"Error evaluating deterministic assignment for {action_config.variable}='{action_config.value}': {e}")
                reward -= 100.0
                terminated = True
                applied_successfully = False
            return reward, terminated, applied_successfully

        elif isinstance(action_config, pyast_config.NondeterministicAssignmentActionConfig):
            assign_key = f"assign_{action_config.variable}"
            if assign_key in agent_action:
                self.state[action_config.variable] = agent_action[assign_key]
            else:
                print(f"Warning: Nondeterministic assignment for '{action_config.variable}' (key '{assign_key}') not found in agent action. Defaulting to current value.")
                # This might indicate an incomplete action by the agent, penalize.
                reward -= 50.0
                # Do not terminate here, as it might be a valid choice to not assign.
                # However, if the problem requires it, this might need to be `terminated = True`.
                # For now, we assume it's okay to skip, but with a penalty.
            return reward, terminated, applied_successfully

        elif isinstance(action_config, pyast_config.ActionConstraintConfig):
            constraint_met = self._evaluate_expression(action_config.formula, is_constraint=True)

            # Try to find metadata for this constraint if it's a predicate call
            predicate_name_match = re.match(r'^\s*(\w+)\s*\(', action_config.formula)
            if predicate_name_match:
                pred_name = predicate_name_match.group(1)
                metadata = self._get_predicate_metadata(pred_name)
            else:
                # Default metadata for generic constraints
                metadata = {'type': 'safety', 'penalty': '-100.0', 'on_failure': 'terminate:status=failure'}

            if not constraint_met:
                reward, terminated, truncated = self._apply_meta_effect(metadata, reward, terminated, False, is_failure_case=True)
                # If metadata didn't specify termination/truncation, and it's not 'continue', then it's a failure to apply
                if not terminated and not truncated and metadata.get('on_failure', '').split(':')[0] == 'continue':
                    applied_successfully = True # Constraint failed but we are told to continue
                else:
                    applied_successfully = False # Constraint failed and we should terminate/truncate
            else:
                # If constraint is met, apply 'on_success' effects if any
                if metadata.get('on_success') and metadata.get('on_success').split(':')[0] != 'continue':
                    reward, terminated, truncated = self._apply_meta_effect(metadata, reward, terminated, truncated, is_failure_case=False)
                applied_successfully = True # Constraint met, so action was successfully applied

            return reward, terminated, applied_successfully

        elif isinstance(action_config, pyast_config.CallHybridProgramConfig):
            # This generator assumes controlFunc is the primary action selection.
            # If CallHybridProgramConfig appears here, it's an unsupported nested HP call within controlFunc.
            print(f"Warning: CallHybridProgramConfig '{action_config.hp_name}' encountered in _execute_action_config. Not fully supported yet.")
            reward -= 50.0
            applied_successfully = False
            return reward, terminated, applied_successfully

        else:
            print(f"Error: Unknown action config type: {type(action_config)}")
            reward -= 200.0
            terminated = True
            applied_successfully = False
            return reward, terminated, applied_successfully

    def _apply_control_action(self, action):
        """
        Adds the agent's chosen action by traversing the control function's AST.
        """
        # For a simple Discrete action space, the agent_action is just an integer (choice index).
        # We need to map this to the expected dictionary format for _execute_action_config.
        # This assumes the top-level action config is a ChoiceActionConfig.
        agent_action_dict = {f"choice_{self._control_func_action_config.level_name}": action}

        reward, terminated, applied_successfully = self._execute_action_config(self._control_func_action_config, agent_action_dict)

        # Apply t := 0; after controlFunc, as specified in the problem loop body
        if applied_successfully: # Only reset t if the action was valid and applied
            self.state['t'] = 0.0

        return reward, terminated, applied_successfully


    def _split_top_level_and_local(self, formula_str: str) -> List[str]:
        """
        Splits a formula string by top-level '&' operators, respecting brace nesting,
        using the utility from pyast_config.
        """
        return pyast_config._split_by_operator_outside_parens(formula_str, '&')


    def _populate_cisb(self):
        """
        Populates the Contrastive Initial State Buffer (CISB) with initial states
        that satisfy the problem's precondition, using Z3 for verification.
        For strict preconditions (like the current example), this will result
        in the buffer being filled with identical states.
        """
        print(f"Populating Contrastive Initial State Buffer (CISB) with {self._cisb_size} states...")
        current_cisb_count = 0
        generated_states_set = set() # To track unique states

        precondition_formula = self._problem_config.precondition.formula
        print(f"DEBUG: _populate_cisb - Value of self._problem_config.precondition.formula before Z3 call: '{precondition_formula}'")
        if not precondition_formula:
            raise ValueError("CRITICAL ERROR: ProblemConfig.precondition.formula is empty. Cannot populate CISB without a valid precondition.")

        # Attempt to find one satisfying initial state without any guiding hints.
        # This will find the unique solution if the precondition is a conjunction of equalities.
        # If the precondition allows for ranges, this will find one arbitrary solution.
        initial_solution = self._solve_initial_state_with_z3(precondition_formula, hint_values=None)

        if not initial_solution:
            raise ValueError(f"CRITICAL ERROR: Precondition '{precondition_formula}' is unsatisfiable. Cannot populate CISB.")

        # Create a canonical representation of this single state for uniqueness check (though it will be unique initially)
        # and to ensure all necessary keys are present.
        all_relevant_keys = set(self._state_keys)
        for const_decl in self._definitions_config.constants:
            all_relevant_keys.add(const_decl.name)

        canonical_initial_state_representation = tuple(sorted([
            (k, float(initial_solution.get(k, 0.0))) for k in all_relevant_keys
        ]))

        # Add the unique initial solution to the CISB.
        if canonical_initial_state_representation not in generated_states_set:
            self._cisb.append(initial_solution)
            generated_states_set.add(canonical_initial_state_representation)
            current_cisb_count += 1

        # Since the current precondition is a set of strict equalities, there's only one
        # unique satisfying state. We fill the rest of the CISB with this same state.
        # If the precondition were less strict, a more complex sampling strategy
        # using hint_values would be needed here to find diverse states.
        while current_cisb_count < self._cisb_size:
            self._cisb.append(initial_solution)
            current_cisb_count += 1

        print(f"CISB population complete with {current_cisb_count} states.")
        if not self._cisb:
            print("CRITICAL ERROR: CISB is empty after population attempt. This should not happen if precondition is satisfiable.")


    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0

        if not self._cisb:
            print("Error: CISB is empty. Attempting to generate a single initial state using Z3 as fallback.")
            # Define fallback hints for a single Z3 call
            fallback_hint_values = {
                'x': 0.0, 'y': 0.0, 'v': 0.0, 'dx': 1.0, 'dy': 0.0, 'w': 0.0,
                'r': 1.0, 'xo': 5.0, 'yo': 0.0, 't': 0.0, 'ep': 1.0, 'b': 1.0, 'A': 0.0, 'W': 0.0
            }
            # Use the precondition formula from ProblemConfig, which should now be checked
            z3_base_solution = self._solve_initial_state_with_z3(self._problem_config.precondition.formula, fallback_hint_values)

            if z3_base_solution:
                final_initial_state = z3_base_solution
            else:
                print("CRITICAL ERROR: Z3 also failed to find a fallback initial state. Using hardcoded defaults which might violate preconditions.")
                # Truly hardcoded defaults if Z3 fails completely
                final_initial_state = {
                    'x': 0.0, 'y': 0.0, 'v': 0.0, 'a': 0.0, 'dx': 1.0, 'dy': 0.0, 'w': 0.0,
                    'r': 1.0, 'xo': 5.0, 'yo': 0.0, 't': 0.0, 'ep': 1.0, 'b': 1.0, 'A': 0.0, 'W': 0.0
                }
        else:
            # Sample a random state from the Contrastive Initial State Buffer
            # Use self.np_random for consistent seeding
            chosen_initial_state_entry = self.np_random.choice(self._cisb)
            final_initial_state = chosen_initial_state_entry.copy()

        # Apply the chosen initial state (from CISB or fallback)
        for var_name, value in final_initial_state.items():
            if var_name in self.state:
                self.state[var_name] = value
            elif var_name in self.constants:
                self.constants[var_name] = value

        self.last_failing_precondition_formula = None # Reset for the new episode

        observation = self._get_obs()
        info = self._get_info()
        return observation, info

    def step(self, action):
        self.current_step += 1
        reward = 0.0
        terminated = False
        truncated = False
        info = self._get_info()

        # --- Initial Loop Invariant Check (before control/dynamics) ---
        loop_invariant_formula_kx = self._problem_config.loop_execution.loop_invariant_formula
        expanded_loop_inv = self._expand_formula(loop_invariant_formula_kx, self._definitions_config)
        loop_inv_conjuncts = self._split_top_level_and_local(expanded_loop_inv)

        loop_invariant_satisfied_pre_step = True
        for conj_str in loop_inv_conjuncts:
            conjunct_result = self._evaluate_expression(conj_str, is_constraint=True)
            predicate_name_match = re.match(r'^\s*(\w+)\s*\(', conj_str)
            pred_name = predicate_name_match.group(1) if predicate_name_match else None
            metadata = self._get_predicate_metadata(pred_name) if pred_name else {}

            if not conjunct_result:
                loop_invariant_satisfied_pre_step = False
                # Apply failure effects
                reward, terminated, truncated = self._apply_meta_effect(
                    metadata or {'type': 'safety', 'penalty': '-100.0', 'on_failure': 'terminate:status=failure'},
                    reward, terminated, truncated, is_failure_case=True
                )
                if terminated or truncated:
                    return self._get_obs(), reward, terminated, truncated, info
            else:
                # Apply success effects if any (only if action is not 'continue')
                if metadata.get('on_success') and metadata.get('on_success').split(':')[0] != 'continue':
                    reward, terminated, truncated = self._apply_meta_effect(metadata, reward, terminated, truncated, is_failure_case=False)
                    if terminated or truncated:
                        return self._get_obs(), reward, terminated, truncated, info

        if not loop_invariant_satisfied_pre_step:
            # Fallback for general loopinv failure if not caught by specific conjuncts
            reward, terminated, truncated = self._apply_meta_effect(
                {'type': 'safety', 'penalty': '-500.0', 'on_failure': 'terminate:status=failure'},
                reward, terminated, truncated, is_failure_case=True
            )
            return self._get_obs(), reward, terminated, truncated, info # Terminate early


        # --- 1. Apply Control Function (Discrete Action) ---
        control_reward, control_terminated, control_applied = self._apply_control_action(action)
        reward += control_reward
        terminated = terminated or control_terminated

        if not control_applied:
            # Invalid action chosen or constraint violated during control phase
            return self._get_obs(), reward, terminated, truncated, info

        # --- 2. Simulate Differential Equations (Continuous Dynamics) ---
        # Prepare initial state for ODE solver
        y0 = np.array([self.state[var_name] for var_name in self._ode_state_map.keys()], dtype=np.float64)

        ep_val = self.constants['ep'] # Get the 'ep' constant value
        t_span = (0, ep_val) # Integrate up to time limit 'ep'

        ode_terminated_by_event = False

        if ep_val <= 0:
            # If ep is non-positive, no meaningful time evolution occurs.
            self.state['t'] = 0.0 # Time does not advance or is reset
            # This is a violation of 'ep > 0' from bounds() predicate.
            # If the CISB is populated correctly, ep should always be > 0.
            # This case might indicate an horrific with CISB or a very unusual state.
            print(f"Warning: 'ep' constant is non-positive ({ep_val}). Skipping ODE integration and penalizing.")
            phys_valid_meta = self._get_predicate_metadata('bounds') # Use bounds predicate for this check
            reward, terminated, truncated = self._apply_meta_effect(
                phys_valid_meta or {'type': 'safety', 'penalty': '-200.0', 'on_failure': 'terminate:status=failure'},
                reward, terminated, truncated, is_failure_case=True
            )
            ode_terminated_by_event = True # Treat as terminated by event
        else:
            try:
                solution = solve_ivp(
                    self._ode_func_compiled,
                    t_span,
                    y0,
                    events=self._ode_events_compiled,
                    dense_output=True,
                    rtol=1e-6, atol=1e-8
                )

                if solution.status == 0: # Integration completed successfully
                    # Update state with final values from ODE solver
                    final_y = solution.y[:, -1]
                    for i, var_name in enumerate(self._ode_state_map.keys()):
                        self.state[var_name] = final_y[i]
                elif solution.status == 1: # An event occurred
                    # Update state to the point where the event occurred
                    if solution.y_events and solution.y_events[0].size > 0:
                        final_y = solution.y_events[0][0]
                    else:
                        final_y = solution.y[:, -1] # Fallback if event data is missing

                    for i, var_name in enumerate(self._ode_state_map.keys()):
                        self.state[var_name] = final_y[i]

                    # Check which event terminated it
                    # If v < 0, it's a domain violation -> termination
                    if self.state.get('v', 0.0) < -self.equality_tolerance: # Use tolerance for v < 0 check
                        phys_valid_meta = self._get_predicate_metadata('isPhysicallyValid')
                        reward, terminated, truncated = self._apply_meta_effect(
                            phys_valid_meta or {'type': 'safety', 'penalty': '-100.0', 'on_failure': 'truncate:status=failure'}, # Default to truncate as per problem description example
                            reward, terminated, truncated, is_failure_case=True
                        )
                        ode_terminated_by_event = True
                    # If t > ep, it means the time limit was reached, which is expected behavior.
                    # No additional penalty for that specific event, as it's part of the evolution domain.
                else: # Integration failed (status 2)
                    reward, terminated, truncated = self._apply_meta_effect(
                        {'type': 'safety', 'penalty': '-150.0', 'on_failure': 'terminate:status=failure'},
                        reward, terminated, truncated, is_failure_case=True
                    )
                    ode_terminated_by_event = True

            except Exception as e:
                reward, terminated, truncated = self._apply_meta_effect(
                    {'type': 'safety', 'penalty': '-200.0', 'on_failure': 'terminate:status=failure'},
                    reward, terminated, truncated, is_failure_case=True
                )
                ode_terminated_by_event = True

        terminated = terminated or ode_terminated_by_event
        truncated = truncated or ode_terminated_by_event # If ODE event truncates, propagate

        # If ODE integration or control action led to termination/truncation, return early
        if terminated or truncated:
            return self._get_obs(), reward, terminated, truncated, info

        # --- 3. Evaluate Loop Invariant and Apply Rewards/Penalties/Shaping ---
        # Re-evaluate loop invariant after ODEs
        loop_invariant_satisfied_post_ode = True
        for conj_str in loop_inv_conjuncts: # Reuse loop_inv_conjuncts from earlier
            conjunct_result = self._evaluate_expression(conj_str, is_constraint=True)
            predicate_name_match = re.match(r'^\s*(\w+)\s*\(', conj_str)
            pred_name = predicate_name_match.group(1) if predicate_name_match else None
            metadata = self._get_predicate_metadata(pred_name) if pred_name else {}

            if not conjunct_result:
                loop_invariant_satisfied_post_ode = False
                # Apply failure effects
                reward, terminated, truncated = self._apply_meta_effect(
                    metadata or {'type': 'safety', 'penalty': '-100.0', 'on_failure': 'terminate:status=failure'},
                    reward, terminated, truncated, is_failure_case=True
                )
                if terminated or truncated:
                    return self._get_obs(), reward, terminated, truncated, info
            else:
                # Apply success effects if any (only if action is not 'continue')
                if metadata.get('on_success') and metadata.get('on_success').split(':')[0] != 'continue':
                    reward, terminated, truncated = self._apply_meta_effect(metadata, reward, terminated, truncated, is_failure_case=False)
                    if terminated or truncated:
                        return self._get_obs(), reward, terminated, truncated, info

        if not loop_invariant_satisfied_post_ode:
            # Fallback for general loopinv failure if not caught by specific conjuncts
            reward, terminated, truncated = self._apply_meta_effect(
                {'type': 'safety', 'penalty': '-300.0', 'on_failure': 'terminate:status=failure'},
                reward, terminated, truncated, is_failure_case=True
            )
            if terminated or truncated:
                return self._get_obs(), reward, terminated, truncated, info

        # --- Reward Shaping (from progress predicates) ---
        # isWellformedDir (progress type, shape: -((dx^2 + dy^2) - 1)^2)
        wellformed_meta = self._get_predicate_metadata('isWellformedDir')
        if wellformed_meta.get('type') == 'progress' and 'shape' in wellformed_meta:
            shape_value = self._evaluate_shape_expression(wellformed_meta['shape'])
            reward += shape_value # Add to reward

        # isEfficient (progress type, shape: v > 0 ? 1/v : v)
        efficient_meta = self._get_predicate_metadata('isEfficient')
        if efficient_meta.get('type') == 'progress' and 'shape' in efficient_meta:
            shape_value = self._evaluate_shape_expression(efficient_meta['shape'])
            reward += shape_value # Add to reward


        # --- 4. Check Problem Postcondition (ensure) ---
        # This is the goal of the problem.
        postcondition_formula_kx = self._problem_config.postcondition.formula
        postcondition_satisfied = self._evaluate_expression(postcondition_formula_kx, is_constraint=True)
        ensure_meta = self._get_predicate_metadata('ensure') # Get metadata for 'ensure' predicate

        if postcondition_satisfied:
            reward, terminated, truncated = self._apply_meta_effect(
                ensure_meta, reward, terminated, truncated, is_failure_case=False
            )
        else:
            reward, terminated, truncated = self._apply_meta_effect(
                ensure_meta, reward, terminated, truncated, is_failure_case=True
            )

            # Also apply shape if available for the postcondition (ensure)
            if ensure_meta.get('type') == 'progress' and 'shape' in ensure_meta:
                shape_value = self._evaluate_shape_expression(ensure_meta['shape'])
                reward += shape_value # Add to reward

        # --- 5. Check Truncation ---
        if self.current_step >= self.max_episode_steps:
            truncated = True
            # Small penalty for truncating without reaching goal, unless already terminated by ensure failure
            if not terminated and not postcondition_satisfied:
                reward -= 10.0

        observation = self._get_obs()
        info = self._get_info()
        return observation, reward, terminated, truncated, info

    def render(self):
        if self.render_mode == "human":
            # In a real environment, this would draw the robot, obstacle, etc.
            # For now, just print state for demonstration.
            print(f"Render: Step={self.current_step}, State={self.state}")

    def close(self):
        # No specific resources to close for this environment
        pass

In [2]:
import gymnasium as gym
import numpy as np
import math
import sys
import os

# Import the environment class
# Import AST classes from pyast_config. This assumes pyast_config.py exists.
# If pyast_config.py is not available, you would need to mock these classes
# as shown in the thought process.
from pyast_config import (ActionType, ChoiceActionConfig, SequenceActionConfig, DeterministicAssignmentActionConfig, NondeterministicAssignmentActionConfig, ActionConstraintConfig, CallHybridProgramConfig, HybridProgramStepConfig, GymEnvironmentConfig, DefinitionsConfig, VariablesConfig, ProblemConfig, ConstantDefinitionConfig, FunctionDefinitionConfig, PredicateDefinitionConfig, HybridProgramConfig, ControlFuncConfig, StepFuncConfig, ODEConfig, EvolutionDomainConfig, VariableDeclarationConfig, PreconditionConfig, PostconditionConfig, LoopExecutionConfig)


def run_environment_test(env_seed=42, num_steps=30):
    print(f"--- Starting Environment Test with seed {env_seed} ---")

    # Instantiate the environment, passing the seed to the constructor
    # This ensures reproducible CISB population.
    env = GeneratedKXEnv(render_mode="human", cisb_size=10, cisb_populate_attempts=50, seed=env_seed)

    print("\n--- CISB Check ---")
    if env._cisb:
        print(f"CISB populated successfully with {len(env._cisb)} states.")
        # Optionally, print a sample state from CISB
        print(f"Sample CISB entry (first 5 keys): {dict(list(env._cisb[0].items())[:5])}...")
        for i, cisb_entry in enumerate(env._cisb):
            print(f"CISB Entry {i+1}: {cisb_entry}")
    else:
        print("CRITICAL ERROR: CISB is empty. This indicates a problem with _populate_cisb or precondition satisfiability.")
        env.close()
        return # Exit if CISB is not populated

    # Reset the environment for the first episode.
    # This also seeds the environment's internal RNG for episode dynamics.
    initial_obs, initial_info = env.reset(seed=env_seed)
    print(f"\n--- Initial State after reset (seed {env_seed}) ---")
    print(f"Observation: {initial_obs}")
    print(f"Info: {initial_info}")

    # Evaluate precondition for the initial state
    precondition_formula = env._problem_config.precondition.formula
    expanded_precondition = env._expand_formula(precondition_formula, env._definitions_config)
    is_initial_state_valid = env._evaluate_expression(expanded_precondition, is_constraint=True)
    print(f"Precondition '{precondition_formula}' evaluated to: {is_initial_state_valid}")
    assert is_initial_state_valid, "Initial state does not satisfy precondition!"
    print("Assertion Passed: Initial state satisfies precondition.")

    print("\n--- Stepping through the environment ---")
    current_action_choice = 0 # Cycle through action choices 0, 1, 2

    for step_count in range(num_steps):
        print(f"\n--- Step {step_count + 1} ---")

        action = {}
        # Get current state values for constructing actions
        current_v = env.state['v']
        current_x = env.state['x']
        current_y = env.state['y']
        current_W = env.constants['W']
        current_b = env.constants['b']
        current_A = env.constants['A']
        current_ep = env.constants['ep']

        if current_action_choice == 0: # a = -b (decelerate)
            action['choice_ChoiceLevel_0'] = 0
            print(f"Agent chooses Action 0: a = -b (Decelerate)")
        elif current_action_choice == 1: # v = 0 -> a = 0, w = 0 (stop/idle)
            action['choice_ChoiceLevel_0'] = 1
            print(f"Agent chooses Action 1: v = 0 -> a = 0, w = 0 (Stop/Idle)")
            # This action has a constraint 'v = 0'. If v is not 0, it will be penalized.
            # We'll just try it regardless for testing purposes.
        elif current_action_choice == 2: # a = A, nondeterministic w, r, xo, yo (accelerate/turn with obstacle avoidance)
            action['choice_ChoiceLevel_0'] = 2
            print(f"Agent chooses Action 2: a = A, nondeterministic w, r, xo, yo (Accelerate/Turn)")

            # --- Constructing Nondeterministic Assignments for Action 2 ---
            # Constraints: -W<=w & w<=W, r!=0 & r*w = v, abs(x-xo) > admissibleSeparation(v) | abs(y-yo) > admissibleSeparation(v)

            # Strategy for w and r:
            # 1. Ensure r*w = v.
            # 2. Ensure -W <= w <= W.
            # 3. Ensure r != 0.

            assigned_w = 0.0
            assigned_r = 1.0 # Default non-zero r

            if abs(current_v) < env.equality_tolerance: # If v is essentially zero
                assigned_w = env.np_random.uniform(-current_W, current_W) if current_W > env.equality_tolerance else 0.0
                assigned_r = env.np_random.uniform(0.1, 5.0) # Any non-zero r
            else: # If v is non-zero
                # Try to find w such that -W <= w <= W and then calculate r = v/w
                # To guarantee a solution, we might need to adjust r or w.
                # Let's try to set w to a value that is likely within bounds and then calculate r.
                # If v is small, w can be small. If v is large, w needs to be appropriately scaled.
                # A simple approach: target w to be a fraction of W, then calculate r.
                if current_W > env.equality_tolerance:
                    # Target w to be within [-0.5*W, 0.5*W]
                    target_w_magnitude = env.np_random.uniform(0.1 * current_W, 0.5 * current_W)
                    assigned_w = np.sign(current_v) * target_w_magnitude # Match sign of v
                    # Ensure assigned_w is not too close to zero if v is non-zero, to avoid large r
                    if abs(assigned_w) < env.equality_tolerance * 10 and abs(current_v) > env.equality_tolerance:
                        assigned_w = np.sign(current_v) * (0.1 * current_W if current_W > env.equality_tolerance else 1.0)

                    if abs(assigned_w) > env.equality_tolerance:
                        assigned_r = current_v / assigned_w
                    else: # Fallback if assigned_w is still too small, force r to be large
                        assigned_r = current_v / (env.equality_tolerance * np.sign(current_v) if np.sign(current_v) != 0 else 1.0)
                        assigned_w = current_v / assigned_r # Recalculate w to satisfy r*w=v

                else: # W is effectively zero, so w must be zero. This implies v must be zero for r*w=v
                    assigned_w = 0.0
                    if abs(current_v) > env.equality_tolerance:
                        print(f"Warning: W is zero but v={current_v}. Constraint 'r*w=v' will likely fail for Action 2.")
                    assigned_r = env.np_random.uniform(0.1, 5.0) # Still need r != 0

            # Final check for r != 0
            if abs(assigned_r) < env.equality_tolerance:
                assigned_r = 1.0 # Ensure r is not zero

            action['assign_w'] = assigned_w
            action['assign_r'] = assigned_r

            # For xo, yo: make them far from x, y to satisfy admissibleSeparation
            # This is a heuristic to try to pass the safety constraint.
            # admissibleSeparation depends on v, b, A, ep.
            admissible_sep_val = 0.0
            try:
                admissible_sep_val = env.functions['admissibleSeparation'](current_v)
                # Ensure a minimum separation even if admissible_sep_val is small or negative (due to constants)
                if admissible_sep_val <= 0:
                    admissible_sep_val = 1.0
            except Exception as e:
                print(f"Warning: Error calculating admissibleSeparation: {e}. Using default 1.0.")
                admissible_sep_val = 1.0

            # Place obstacle far away
            offset_x = env.np_random.uniform(admissible_sep_val + 5.0, admissible_sep_val + 10.0)
            offset_y = env.np_random.uniform(admissible_sep_val + 5.0, admissible_sep_val + 10.0)
            action['assign_xo'] = current_x + offset_x * env.np_random.choice([-1, 1])
            action['assign_yo'] = current_y + offset_y * env.np_random.choice([-1, 1])

            print(f"  Assigned w={action['assign_w']:.4f}, r={action['assign_r']:.4f}, xo={action['assign_xo']:.4f}, yo={action['assign_yo']:.4f}")

        # Perform the step
        obs, reward, terminated, truncated, info = env.step(action)

        print(f"  Observation: {obs}")
        print(f"  Reward: {reward:.4f}")
        print(f"  Terminated: {terminated}, Truncated: {truncated}")
        print(f"  Info (current state): {info}")

        # --- Basic Invariant Assertions after each step ---
        # Check if v >= 0 (from isPhysicallyValid and evolution domain)
        assert info['v'] >= -env.equality_tolerance, f"Assertion Failed: Velocity became negative: v={info['v']:.4f}"
        # Check if dx^2 + dy^2 approx 1 (from isWellformedDir)
        dir_mag_sq = info['dx']**2 + info['dy']**2
        assert env._approx_eq(dir_mag_sq, 1), f"Assertion Failed: Direction vector not normalized: dx^2+dy^2={dir_mag_sq:.4f}"
        # Check if r != 0
        assert abs(info['r']) > env.equality_tolerance, f"Assertion Failed: r became zero: r={info['r']:.4f}"
        print("  Assertions Passed: v >= 0, dx^2+dy^2 approx 1, r != 0.")


        if terminated or truncated:
            print(f"Episode finished at step {step_count + 1}. Status: Terminated={terminated}, Truncated={truncated}")
            break

        # Cycle through action types for variety
        current_action_choice = (current_action_choice + 1) % 3
    

    env.close()
    print("\n--- Environment Test Finished ---")

# Run the test
if __name__ == "__main__":
    run_environment_test(num_steps=20) # Run for 20 steps

--- Starting Environment Test with seed 42 ---
DEBUG: GeneratedKXEnv.__init__ - Initializing self._problem_config.precondition.formula to: 'x=0 & y=0 & v=0 & dx=1 & dy=0 & t=0 & r=1 & xo=5 & yo=0 & ep=1 & b=1 & A=0 & W=0'
Populating Contrastive Initial State Buffer (CISB) with 10 states...
DEBUG: _populate_cisb - Value of self._problem_config.precondition.formula before Z3 call: 'x=0 & y=0 & v=0 & dx=1 & dy=0 & t=0 & r=1 & xo=5 & yo=0 & ep=1 & b=1 & A=0 & W=0'
DEBUG: _solve_initial_state_with_z3 - Received precondition_kx: 'x=0 & y=0 & v=0 & dx=1 & dy=0 & t=0 & r=1 & xo=5 & yo=0 & ep=1 & b=1 & A=0 & W=0'
DEBUG: _kx_expr_to_z3_formula - Initial kx_expr: 'x=0 & y=0 & v=0 & dx=1 & dy=0 & t=0 & r=1 & xo=5 & yo=0 & ep=1 & b=1 & A=0 & W=0'
DEBUG: _kx_expr_to_z3_formula - After strip: 'x=0 & y=0 & v=0 & dx=1 & dy=0 & t=0 & r=1 & xo=5 & yo=0 & ep=1 & b=1 & A=0 & W=0'
DEBUG: _kx_expr_to_z3_formula - After _expand_for_z3: 'x=0 & y=0 & v=0 & dx=1 & dy=0 & t=0 & r=1 & xo=5 & yo=0 & ep=1 & b=1 & A=

KeyError: 'v'